# Decision Tree Classification – Heart Disease
**Objective:** Predict heart disease using Decision Tree Classification.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

df = pd.read_excel(r'C:\Users\Admin\OneDrive\Desktop\Data Science ExcelR\Assignments\Assignment 11\heart_disease.xlsx')
print('Shape:', df.shape)
display(df.head())

## 1. EDA

In [ ]:
print(df.info())
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
display(df.describe())

df.hist(figsize=(12,8), bins=20)
plt.tight_layout(); plt.show()

plt.figure(figsize=(12,5))
sns.boxplot(data=df.select_dtypes(include=np.number))
plt.xticks(rotation=90); plt.show()

plt.figure(figsize=(10,7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix'); plt.show()

## 2. Feature Engineering

In [ ]:
# num: 0 = no disease, 1-4 = disease stages
df['target'] = (df['num'] > 0).astype(int)
X = df.drop(columns=['num','target']).copy()
y = df['target']

# Handle missing values
X = X.fillna(X.median(numeric_only=True))

print('Classes:\n', y.value_counts())
sns.countplot(x=y)
plt.title('Class Distribution'); plt.show()

## 3. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 4. Decision Tree Model

In [ ]:
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, pred))
print('Accuracy :', round(accuracy_score(y_test,pred),4))
print('Precision:', round(precision_score(y_test,pred),4))
print('Recall   :', round(recall_score(y_test,pred),4))
print('F1 Score :', round(f1_score(y_test,pred),4))
print('ROC-AUC  :', round(roc_auc_score(y_test,prob),4))

## 5. Hyperparameter Tuning

In [ ]:
params = {
    'max_depth':[3,5,7,10,None],
    'min_samples_split':[2,5,10],
    'criterion':['gini','entropy']
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42), params, cv=5, scoring='f1'
)
grid.fit(X_train,y_train)
print('Best Parameters:', grid.best_params_)

## 6. Final Evaluation

In [ ]:
best = grid.best_estimator_
pred = best.predict(X_test)
prob = best.predict_proba(X_test)[:,1]

print(classification_report(y_test,pred))
print('Accuracy :', round(accuracy_score(y_test,pred),4))
print('Precision:', round(precision_score(y_test,pred),4))
print('Recall   :', round(recall_score(y_test,pred),4))
print('F1 Score :', round(f1_score(y_test,pred),4))
print('ROC-AUC  :', round(roc_auc_score(y_test,prob),4))

## 7. Decision Tree & Feature Importance

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(best, feature_names=X.columns, class_names=['No Disease','Disease'], filled=True, max_depth=3)
plt.show()

importance = pd.Series(best.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importance)
importance.plot(kind='bar', figsize=(10,5), title='Feature Importance')
plt.show()

## Conclusion
The Decision Tree was trained to classify patients into no-disease and disease groups. Missing values were handled, the data was split 80/20, and the model was evaluated using Accuracy, Precision, Recall, F1-score and ROC-AUC. GridSearchCV optimized the tree using depth, split size and criterion. Feature importance and the tree structure help interpret the model.

## Interview Questions
**1. Common hyperparameters:** `max_depth` controls tree depth; `min_samples_split` controls when a node can split; `min_samples_leaf` controls minimum observations in a leaf; `criterion` selects Gini or Entropy. These parameters help control overfitting.

**2. Label vs One-hot encoding:** Label encoding maps categories to numbers (e.g., Male=0, Female=1). One-hot encoding creates separate 0/1 columns for each category. One-hot avoids implying an order between nominal categories.